<a href="https://colab.research.google.com/github/Alozysr/decoder-llm-from-scratch/blob/main/dil_modeli_denemesi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
import os
import shutil

# Bağlı olup olmadığını kontrol et ve eğer bağlıysa kaldır
if os.path.ismount('/content/drive'):
    try:
        drive.flush_and_unmount()
        print("Google Drive bağlantısı kaldırıldı.")
    except Exception as e:
        print(f"Bağlantı kaldırılamadı: {e}")

# /content/drive dizini mevcutsa ve boş değilse, sil ve yeniden oluştur
if os.path.exists('/content/drive') and os.listdir('/content/drive'):
    print("'/content/drive' dizininde kalan dosyalar bulundu, temizleniyor...")
    try:
        shutil.rmtree('/content/drive')
        print("'/content/drive' dizini başarıyla temizlendi.")
    except Exception as e:
        print(f"'/content/drive' dizini temizlenirken hata oluştu: {e}. Lütfen çalışma zamanını yeniden başlatmayı deneyin.")

# Dizinin var olduğundan emin ol (temizleme sonrası yeniden oluşturulması gerekebilir)
if not os.path.exists('/content/drive'):
    os.makedirs('/content/drive')

# Google Drive'ı yeniden bağla
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
import os
project_dir = "/content/drive/MyDrive/mini-llm-tr"
os.makedirs(project_dir, exist_ok=True)

In [ ]:
# 1. Gerekli kütüphaneler
!pip install tiktoken datasets numpy -q


In [ ]:
# 2. Veri setini indirme, kurgu (fiction) olanları filtrele
from datasets import load_dataset
import json

dataset = load_dataset("sedthh/gutenberg_english")
print(f"Ham dataset boyutu: {len(dataset['train'])}")

# ---- Kurgu/roman filtresi ----
def is_fiction(example):
    try:
        meta = json.loads(example["METADATA"])
        text = (meta.get("bookshelves", "") + " " + meta.get("subjects", "")).lower()
        fiction_keywords = ["fiction", "novel", "romance", "adventure", "mystery", "gothic"]
        return any(kw in text for kw in fiction_keywords)
    except (json.JSONDecodeError, TypeError):
        return False

dataset["train"] = dataset["train"].filter(is_fiction)
print(f"Kurgu filtresinden sonra: {len(dataset['train'])}")

# ---- Kontrol: ilk örnek artık gerçekten bir roman mı? ----
print(dataset["train"][0]["TEXT"][:1500])

In [ ]:
total_rows = len(dataset["train"])
print(f"Filtrelenmiş toplam kurgu kitabı: {total_rows}")

n_select =  total_rows # istediğin sayıya göre ayarla
dataset["train"] = dataset["train"].shuffle(seed=42).select(range(n_select))
print(f"Seçilen kitap sayısı: {len(dataset['train'])}")

In [ ]:
import torch
torch.cuda.empty_cache()

In [ ]:
import tiktoken
import numpy as np
import os

print("CPU çekirdek sayısı:", os.cpu_count())

enc = tiktoken.get_encoding("gpt2")

def process(example):
    ids = enc.encode_ordinary(example["TEXT"])
    ids.append(enc.eot_token)
    return {"ids": ids, "len": len(ids)}   # düzeltildi

# ---- Shuffle + seç ----
total_rows = len(dataset["train"])
print(f"Filtrelenmiş toplam kurgu kitabı: {total_rows}")

n_select = total_rows
dataset["train"] = dataset["train"].shuffle(seed=42).select(range(n_select))
print(f"Seçilen kitap sayısı: {len(dataset['train'])}")

# ---- Gerçek train/validation split oluştur ----
split = dataset["train"].train_test_split(test_size=0.05, seed=42)
dataset = {"train": split["train"], "validation": split["test"]}
print(f"Train: {len(dataset['train'])} kitap, Validation: {len(dataset['validation'])} kitap")

# ---- Her split'i ayrı ayrı tokenize et ----
tokenized = {}
for split_name, dset in dataset.items():
    print(f"\nTokenize ediliyor: {split_name}")   # düzeltildi
    tokenized[split_name] = dset.map(
        process,
        remove_columns=dset.column_names,
        desc=f"tokenizing {split_name}",
        num_proc=os.cpu_count(),
        writer_batch_size=800,
    )

In [ ]:
# validation ve train .bin dosyalarını oluştur
for split_name, dset in tokenized.items():
    arr_len = np.sum(dset["len"], dtype=np.uint64)
    filename = os.path.join(project_dir, f"{split_name}.bin")
    dtype = np.uint16
    arr = np.memmap(filename, dtype=dtype, mode="w+", shape=(arr_len,))

    idx = 0
    for example in dset:
        arr[idx: idx + example["len"]] = example["ids"]
        idx += example["len"]
    arr.flush()
    print(f"{filename} yazıldı, {arr_len} token")

In [ ]:
import os
os.makedirs("mini-llm-tr", exist_ok=True)

In [ ]:
%%writefile /content/drive/MyDrive/mini-llm-tr/model.py
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head, block_size, dropout):
        super().__init__()
        assert n_embd % n_head == 0
        self.c_attn = nn.Linear(n_embd, 3 * n_embd)
        self.c_proj = nn.Linear(n_embd, n_embd)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)
        self.n_head = n_head
        self.n_embd = n_embd
        self.register_buffer("bias", torch.tril(torch.ones(block_size, block_size))
                                     .view(1, 1, block_size, block_size))

    def forward(self, x):
        B, T, C = x.size()
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)

        att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))
        att = att.masked_fill(self.bias[:, :, :T, :T] == 0, float('-inf'))
        att = F.softmax(att, dim=-1)
        att = self.attn_dropout(att)
        y = att @ v
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.resid_dropout(self.c_proj(y))
        return y

class Block(nn.Module):
    def __init__(self, n_embd, n_head, block_size, dropout):
        super().__init__()
        self.ln_1 = nn.LayerNorm(n_embd)
        self.attn = CausalSelfAttention(n_embd, n_head, block_size, dropout)
        self.ln_2 = nn.LayerNorm(n_embd)
        self.mlp = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.GELU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x

class GPT(nn.Module):
    def __init__(self, vocab_size, n_embd=384, n_head=6, n_layer=6, block_size=256, dropout=0.1):
        super().__init__()
        self.block_size = block_size
        self.token_emb = nn.Embedding(vocab_size, n_embd)
        self.pos_emb = nn.Embedding(block_size, n_embd)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([
            Block(n_embd, n_head, block_size, dropout) for _ in range(n_layer)
        ])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)

        # Weight tying: parametre tasarrufu + genelde performans iyileştirmesi
        self.token_emb.weight = self.lm_head.weight

    def forward(self, idx, targets=None):
        B, T = idx.shape
        assert T <= self.block_size, "Sequence uzunluğu block_size'ı aşamaz"
        tok_emb = self.token_emb(idx)
        pos = torch.arange(T, device=idx.device)
        pos_emb = self.pos_emb(pos)
        x = self.drop(tok_emb + pos_emb)
        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, repetition_penalty=1.0, top_k=None):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :]

            # Repetition penalty uygula
            if repetition_penalty != 1.0:
                for b in range(idx.shape[0]):
                    for token in set(idx[b].tolist()):
                        if logits[b, token] > 0:
                            logits[b, token] /= repetition_penalty
                        else:
                            logits[b, token] *= repetition_penalty

            logits = logits / temperature
            if top_k is not None:
                v, _ = torch.topk(logits, top_k)
                logits[logits < v[:, [-1]]] = float('-inf')
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

Overwriting /content/drive/MyDrive/mini-llm-tr/model.py


In [ ]:
%%writefile /content/drive/MyDrive/mini-llm-tr/model.py
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head, block_size, dropout):
        super().__init__()
        assert n_embd % n_head == 0
        self.c_attn = nn.Linear(n_embd, 3 * n_embd)
        self.c_proj = nn.Linear(n_embd, n_embd)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)
        self.n_head = n_head
        self.n_embd = n_embd
        self.register_buffer("bias", torch.tril(torch.ones(block_size, block_size))
                                     .view(1, 1, block_size, block_size))

    def forward(self, x):
        B, T, C = x.size()
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)

        att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))
        att = att.masked_fill(self.bias[:, :, :T, :T] == 0, float('-inf'))
        att = F.softmax(att, dim=-1)
        att = self.attn_dropout(att)
        y = att @ v
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.resid_dropout(self.c_proj(y))
        return y

class Block(nn.Module):
    def __init__(self, n_embd, n_head, block_size, dropout):
        super().__init__()
        self.ln_1 = nn.LayerNorm(n_embd)
        self.attn = CausalSelfAttention(n_embd, n_head, block_size, dropout)
        self.ln_2 = nn.LayerNorm(n_embd)
        self.mlp = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.GELU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x

class GPT(nn.Module):
    def __init__(self, vocab_size, n_embd=384, n_head=6, n_layer=6, block_size=256, dropout=0.1):
        super().__init__()
        self.block_size = block_size
        self.token_emb = nn.Embedding(vocab_size, n_embd)
        self.pos_emb = nn.Embedding(block_size, n_embd)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([
            Block(n_embd, n_head, block_size, dropout) for _ in range(n_layer)
        ])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)

        # Weight tying: parametre tasarrufu + genelde performans iyileştirmesi
        self.token_emb.weight = self.lm_head.weight

    def forward(self, idx, targets=None):
        B, T = idx.shape
        assert T <= self.block_size, "Sequence uzunluğu block_size'ı aşamaz"
        tok_emb = self.token_emb(idx)
        pos = torch.arange(T, device=idx.device)
        pos_emb = self.pos_emb(pos)
        x = self.drop(tok_emb + pos_emb)
        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, repetition_penalty=1.0, top_k=None, top_p=None):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :]

            # Repetition penalty uygula
            if repetition_penalty != 1.0:
                for b in range(idx.shape[0]):
                    for token in set(idx[b].tolist()):
                        if logits[b, token] > 0:
                            logits[b, token] /= repetition_penalty
                        else:
                            logits[b, token] *= repetition_penalty

            logits = logits / temperature

            if top_k is not None:
                v, _ = torch.topk(logits, top_k)
                logits[logits < v[:, [-1]]] = float('-inf')

            if top_p is not None:
                sorted_logits, sorted_indices = torch.sort(logits, descending=True)
                cumulative_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
                # Remove tokens with cumulative probability above the threshold (token with 0 are kept)
                sorted_indices_to_remove = cumulative_probs > top_p
                # Shift the indices to the right to keep at least one token above the threshold
                sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
                sorted_indices_to_remove[..., 0] = 0

                indices_to_remove = sorted_indices[sorted_indices_to_remove]
                logits[:, indices_to_remove] = float('-inf')

            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx


In [ ]:
# Re-import GPT to pick up changes
import sys
import os
project_dir = "/content/drive/MyDrive/mini-llm-tr"
if project_dir not in sys.path:
    sys.path.insert(0, project_dir)
from model import GPT

# Reinitialize the model with the correct vocab_size and other parameters
# Assuming enc, vocab_size, n_embd, n_head, n_layer, block_size, dropout, device are already defined
enc = tiktoken.get_encoding("gpt2")
vocab_size = enc.n_vocab

model = GPT(vocab_size, n_embd, n_head, n_layer, block_size, dropout).to(device)

# Reload the best model weights
model_path = os.path.join(project_dir, "best_model.pt")
if os.path.exists(model_path):
    checkpoint = torch.load(model_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval() # Set to evaluation mode
    print("Model successfully reloaded with updated generate function.")
else:
    print(f"HATA: Model dosyası bulunamadı -> {model_path}")

In [ ]:
import torch
import numpy as np

# ---- Ayarlar ----
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("Kullanılan cihaz:", device)

block_size = 1024  # context length
batch_size = 72   # her adımda kaç örnek işlenecek
max_iters = 20000      # toplam eğitim adımı
eval_interval = 250    # kaç adımda bir validation loss ölçülecek
eval_iters = 50         # validation loss ölçerken kaç batch ortalanacak
learning_rate = 3e-4
weight_decay = 0.1

n_embd = 600
n_head = 10
n_layer = 10
dropout = 0.15

Kullanılan cihaz: cuda


In [ ]:
import os
import numpy as np # numpy'nin burada içe aktarıldığından emin olun
import torch # torch'un burada içe aktarıldığından emin olun

# ---- Veriyi yükle (bir önceki adımda oluşturduğumuz .bin dosyaları) ----
# project_dir değişkeni zaten tanımlanmış olmalı (/content/drive/MyDrive/mini-llm-tr)
train_bin_path = os.path.join(project_dir, "train.bin")
val_bin_path = os.path.join(project_dir, "validation.bin")

print(f"train.bin aranıyor: {train_bin_path}")
if not os.path.exists(train_bin_path):
    raise FileNotFoundError(f"HATA: train.bin dosyası bulunamadı: {train_bin_path}. Lütfen Iyv49e7KU82Q hücresinin başarıyla çalıştığından emin olun.")
else:
    print(f"train.bin bulundu: {train_bin_path}")

print(f"validation.bin aranıyor: {val_bin_path}")
if not os.path.exists(val_bin_path):
    raise FileNotFoundError(f"HATA: validation.bin dosyası bulunamadı: {val_bin_path}. Lütfen Iyv49e7KU82Q hücresinin başarıyla çalıştığından emin olun.")
else:
    print(f"validation.bin bulundu: {val_bin_path}")

train_data = np.memmap(train_bin_path, dtype=np.uint16, mode="r")
val_data = np.memmap(val_bin_path, dtype=np.uint16, mode="r")

def get_batch(split):
    data = train_data if split == "train" else val_data
    # rastgele başlangıç noktaları seç
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([torch.from_numpy(data[i:i+block_size].astype(np.int64)) for i in ix])
    y = torch.stack([torch.from_numpy(data[i+1:i+1+block_size].astype(np.int64)) for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

train.bin aranıyor: /content/drive/MyDrive/mini-llm-tr/train.bin
train.bin bulundu: /content/drive/MyDrive/mini-llm-tr/train.bin
validation.bin aranıyor: /content/drive/MyDrive/mini-llm-tr/validation.bin
validation.bin bulundu: /content/drive/MyDrive/mini-llm-tr/validation.bin


In [ ]:
# ---- Modeli oluştur ----
import tiktoken
import sys
import os

# model.py dosyasının bulunduğu dizini Python yoluna ekle
project_dir = "/content/drive/MyDrive/mini-llm-tr"
if project_dir not in sys.path:
    sys.path.insert(0, project_dir)

from model import GPT # GPT sınıfını model.py dosyasından içe aktar

enc = tiktoken.get_encoding("gpt2")
vocab_size = enc.n_vocab  # 50257

model = GPT(vocab_size, n_embd, n_head, n_layer, block_size, dropout).to(device)
print(sum(p.numel() for p in model.parameters())/1e6, "M parametre")

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

73.7406 M parametre


In [ ]:
print(f"n_embd: {n_embd}, n_head: {n_head}, n_layer: {n_layer}, block_size: {block_size}")

n_embd: 600, n_head: 10, n_layer: 10, block_size: 512


In [ ]:
# ---- Loss ölçme fonksiyonu (train + val) ----
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ["train", "val"]:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split if split == "train" else "val")
            _, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out

In [ ]:
import time
import os

# Eğitim döngüsü
print("Eğitim başlıyor...")
start_time = time.time()

# ---- İyileştirilmiş Ayarlar ----
# 1. Modeli geçici Colab dizini yerine doğrudan Google Drive'a kaydedin
best_model_path = os.path.join(project_dir, "best_model.pt")

# 2. Mixed Precision (AMP) için Scaler (Eğitimi hızlandırır, VRAM tasarrufu sağlar)
scaler = torch.cuda.amp.GradScaler(enabled=(device == 'cuda'))

# 3. Learning Rate Scheduler (Öğrenme oranını eğitim boyunca yavaşça düşürür)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max_iters, eta_min=1e-5)

# ---- Early stopping ayarları ----
patience = 4         # kaç eval_interval boyunca iyileşme olmazsa dur
best_val_loss = float('inf')
patience_counter = 0

#Ana eğitim döngüsü
for iter in range(max_iters):

    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        current_lr = scheduler.get_last_lr()[0]
        print(f"adım {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}, lr: {current_lr:.6f}")

        if losses['val'] < best_val_loss:
            # Val loss iyileşti — en iyi modeli kaydet, sayaç sıfırla
            best_val_loss = losses['val']
            patience_counter = 0
            torch.save({
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'iter': iter,
                'val_loss': best_val_loss,
                'config': {
                    'vocab_size': vocab_size,
                    'n_embd': n_embd,
                    'n_head': n_head,
                    'n_layer': n_layer,
                    'block_size': block_size,
                }
            }, best_model_path)
            print(f"  → yeni en iyi model kaydedildi (val loss: {best_val_loss:.4f})")
        else:
            # İyileşme yok — sayaç artır
            patience_counter += 1
            print(f"  → iyileşme yok ({patience_counter}/{patience})")

            if patience_counter >= patience:
                print(f"\nEarly stopping tetiklendi! adım {iter}'de durduruldu.")
                print(f"En iyi val loss: {best_val_loss:.4f}")
                break

    xb, yb = get_batch("train")

    optimizer.zero_grad(set_to_none=True)

    # Mixed Precision ile Forward Pass
    with torch.autocast(device_type='cuda' if device == 'cuda' else 'cpu', dtype=torch.float16, enabled=(device == 'cuda')):
        logits, loss = model(xb, yb)

    # Scaler ile Backward Pass
    scaler.scale(loss).backward()

    # 4. Gradient Clipping: Gradyan patlamasını önlemek için (Transformerlar için önemlidir)
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

    # Scaler adımı ve LR Scheduler adımı
    scaler.step(optimizer)
    scaler.update()
    scheduler.step()

print("Eğitim tamamlandı!")

Eğitim başlıyor...


/tmp/ipykernel_1565/3160222338.py:13: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(device == 'cuda'))


adım 0: train loss 10.9806, val loss 10.9811, lr: 0.000300
  → yeni en iyi model kaydedildi (val loss: 10.9811)
adım 250: train loss 5.7314, val loss 5.7221, lr: 0.000300
  → yeni en iyi model kaydedildi (val loss: 5.7221)
adım 500: train loss 5.4699, val loss 5.4540, lr: 0.000300
  → yeni en iyi model kaydedildi (val loss: 5.4540)
adım 750: train loss 5.1451, val loss 5.1127, lr: 0.000299
  → yeni en iyi model kaydedildi (val loss: 5.1127)
adım 1000: train loss 4.8760, val loss 4.8628, lr: 0.000298
  → yeni en iyi model kaydedildi (val loss: 4.8628)
adım 1250: train loss 4.6243, val loss 4.6107, lr: 0.000297
  → yeni en iyi model kaydedildi (val loss: 4.6107)
adım 1500: train loss 4.4685, val loss 4.4413, lr: 0.000296
  → yeni en iyi model kaydedildi (val loss: 4.4413)
adım 1750: train loss 4.3135, val loss 4.3110, lr: 0.000295
  → yeni en iyi model kaydedildi (val loss: 4.3110)
adım 2000: train loss 4.2399, val loss 4.2110, lr: 0.000293
  → yeni en iyi model kaydedildi (val loss: 4.2

### 7. Eğitilmiş Modeli Kaydetme
Eğitim tamamlandıktan sonra, eğitilmiş modelin son halini kaydedelim.

In [ ]:
import torch
import os

# 'model' değişkeninin tanımlı olup olmadığını kontrol et
if 'model' in locals() or 'model' in globals():
    # Eğitilmiş modelin son halini kaydet
    final_model_path = os.path.join(project_dir, "final_model.pt")
    torch.save(model.state_dict(), final_model_path)
    print(f"Final model kaydedildi: {final_model_path}")
else:
    print("HATA: 'model' değişkeni bulunamadı. Lütfen modeli tanımlayan ve eğiten önceki hücreleri çalıştırdığınızdan emin olun.")

Final model kaydedildi: /content/drive/MyDrive/mini-llm-tr/final_model.pt


### 8. Metin Oluşturma (Test)
Eğitilmiş modelimizi kullanarak metin oluşturalım.

In [ ]:
import torch
import os

# Kaydedilen en iyi modelin yolu
model_path = os.path.join(project_dir, "best_model.pt")

if os.path.exists(model_path):
    print(f"Model dosyası bulundu, yükleniyor: {model_path}")

    # Checkpoint'i yükle ve modele entegre et
    checkpoint = torch.load(model_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval() # Değerlendirme moduna al
    print("Model başarıyla yüklendi.\n")

    # Başlangıç metni (prompt)
    start_text = "he old carriage stopped in front of the dark mansion. Mr. Darcy stepped out, looking at the broken windows"
    context = torch.tensor(enc.encode(start_text), dtype=torch.long, device=device).unsqueeze(0)

    print("Metin oluşturuluyor...")
    # Doğrudan model.generate kullanılıyor (artık repetition_penalty destekliyor)
    generated_tokens = model.generate(
        context,
        max_new_tokens=750,
        temperature=0.6,
        repetition_penalty=1.2
    )

    # Token'ları tekrar metne dönüştür
    generated_text = enc.decode(generated_tokens[0].tolist())
    print("\n--- Oluşturulan Metin ---")
    print(generated_text)
else:
    print(f"HATA: Model dosyası bulunamadı -> {model_path}")

Model dosyası bulundu, yükleniyor: /content/drive/MyDrive/mini-llm-tr/best_model.pt
Model başarıyla yüklendi.

Metin oluşturuluyor...

--- Oluşturulan Metin ---
he old carriage stopped in front of the dark mansion. Mr. Darcy stepped out, looking at the broken windows and saw for a moment two more words was seen approaching them. He had not been long before this time that night; but he thought it very difficult to be on board with his family’s dinner-party; yet they were all ready to hear him again.
By an hour or so there came a loud hail from behind the door where Mrs. Perven and Miss Martha stood seated, standing by her sidestep. The doctor looked into the passage till she heard the voice distinctly—the sound of the Rector coming up to them—and then wept to see each other’s eyes, as if to say something about what might bring short attention. It seemed like death: the girl was afraid of the dreadfulness in which he spoke and felt when he could get out. “It is Miss Holty!” cried Mr. Wyt

In [ ]:
import torch
import os

# Kaydedilen en iyi modelin yolu
model_path = os.path.join(project_dir, "best_model.pt")

if os.path.exists(model_path):
    print(f"Model dosyası bulundu, yükleniyor: {model_path}")

    # Checkpoint'i yükle ve modele entegre et
    checkpoint = torch.load(model_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval() # Değerlendirme moduna al
    print("Model başarıyla yüklendi.")

    # Başlangıç metni (prompt)
    start_text = "The man"
    context = torch.tensor(enc.encode(start_text), dtype=torch.long, device=device).unsqueeze(0)

    print("Metin oluşturuluyor...")
    # Doğrudan model.generate kullanılıyor (artık repetition_penalty ve top_p destekliyor)
    generated_tokens = model.generate(
        context,
        max_new_tokens=750,
        temperature=0.6,
        repetition_penalty=1.2,
        top_p=0.9 # Using top_p instead of top_k
    )

    # Token'ları tekrar metne dönüştür
    generated_text = enc.decode(generated_tokens[0].tolist())
    print("\n--- Oluşturulan Metin (top_p ile) ---")
    print(generated_text)
else:
    print(f"HATA: Model dosyası bulunamadı -> {model_path}")